# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Same population as the rest of this project: `fact_content_daily_performance`, March 2026, first-half features, `avg_position_h1 > 0` (excludes the known data artifact). Content attributes (`word_count`, `char_count`, `search_volume`) joined from `dim_content`.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

page_level_full = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_impressions) as avg_impressions_h1,
               AVG(gsc_clicks) as avg_clicks_h1,
               AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) as ctr_h1,
               AVG(gsc_avg_position) as avg_position_h1,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) as active_days_h1
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date <= '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT * FROM first_half
""").df()
page_level_full = page_level_full[page_level_full["avg_position_h1"] > 0].copy()

content_features = con.sql(f"""
    SELECT content_hash_id, word_count, char_count, search_volume
    FROM read_parquet('{base}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()
page_level_full = page_level_full.merge(content_features, on="content_hash_id", how="inner")

key_fields = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1",
              "active_days_h1", "word_count", "search_volume"]
print(f"Population: {len(page_level_full):,} pages\n")
print(page_level_full[key_fields].describe(percentiles=[0.5, 0.9, 0.99]).round(2))

print("\nHeavy-tail check - ratio of 99th percentile to median (large ratio = heavy tail):")
for col in key_fields:
    p50 = page_level_full[col].median()
    p99 = page_level_full[col].quantile(0.99)
    ratio = p99 / p50 if p50 > 0 else float("inf")
    print(f"  {col:20s} median={p50:>10.2f}  p99={p99:>12.2f}  p99/median={ratio:>8.1f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Population: 150,533 pages

       avg_impressions_h1  avg_clicks_h1     ctr_h1  avg_position_h1  \
count           150533.00      150533.00  150533.00        150533.00   
mean                59.22           0.18       0.00            15.80   
std                182.46           0.94       0.03            17.68   
min                  1.00           0.00       0.00             0.01   
50%                  8.60           0.00       0.00             8.38   
90%                142.53           0.40       0.01            39.83   
99%                766.80           2.73       0.05            80.58   
max              12428.85         184.23       1.00           310.00   

       active_days_h1  word_count  search_volume  
count       150533.00     99736.0       139178.0  
mean            10.88     2789.68         154.07  
std              5.11     1197.23        2263.69  
min              1.00         0.0            0.0  
50%             14.00      2730.0           10.0  
90%             15

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three intuitive SEO assumptions, tested directly against this portfolio rather than assumed true:

1. **"Worse position means lower CTR."** Correlation between `avg_position_h1` and `ctr_h1`.
2. **"Longer content earns more visibility."** Correlation between `word_count` and `avg_impressions_h1`.
3. **"Consistent visibility associates with better position."** Correlation between `active_days_h1` and `avg_position_h1`.

**Verdicts below are placeholders pending your real run** — filled in as CONFIRMED / OPPOSITE / MIXED / FALSE once you see the actual correlation values, not assumed in advance.

In [2]:
# Signal 1: position vs CTR - lower position number (better rank) should mean higher CTR,
# so a POSITIVE correlation between avg_position_h1 and ctr_h1 would be the OPPOSITE of the
# intuitive assumption (worse position = higher position number = should mean lower CTR,
# i.e. a NEGATIVE correlation is what "CONFIRMED" looks like here).
corr_pos_ctr = page_level_full["avg_position_h1"].corr(page_level_full["ctr_h1"])
print(f"Signal 1: avg_position_h1 vs ctr_h1 correlation = {corr_pos_ctr:.4f}")
print("  Expect NEGATIVE if the assumption holds (worse position -> lower CTR).")
print(f"  VERDICT: {'CONFIRMED' if corr_pos_ctr < -0.1 else 'OPPOSITE' if corr_pos_ctr > 0.1 else 'MIXED/WEAK'} "
      f"(fill in manually after reading the real number - this is a starting heuristic, not the final word)\n")

# Signal 2: word count vs impressions
corr_words_imp = page_level_full["word_count"].corr(page_level_full["avg_impressions_h1"])
print(f"Signal 2: word_count vs avg_impressions_h1 correlation = {corr_words_imp:.4f}")
print("  Expect POSITIVE if the assumption holds (longer content -> more visibility).")
print(f"  VERDICT: {'CONFIRMED' if corr_words_imp > 0.1 else 'OPPOSITE' if corr_words_imp < -0.1 else 'MIXED/WEAK'}\n")

# Signal 3: active days vs position
corr_days_pos = page_level_full["active_days_h1"].corr(page_level_full["avg_position_h1"])
print(f"Signal 3: active_days_h1 vs avg_position_h1 correlation = {corr_days_pos:.4f}")
print("  Expect NEGATIVE if the assumption holds (more consistent visibility -> better/lower position number).")
print(f"  VERDICT: {'CONFIRMED' if corr_days_pos < -0.1 else 'OPPOSITE' if corr_days_pos > 0.1 else 'MIXED/WEAK'}")

Signal 1: avg_position_h1 vs ctr_h1 correlation = -0.0422
  Expect NEGATIVE if the assumption holds (worse position -> lower CTR).
  VERDICT: MIXED/WEAK (fill in manually after reading the real number - this is a starting heuristic, not the final word)

Signal 2: word_count vs avg_impressions_h1 correlation = 0.0805
  Expect POSITIVE if the assumption holds (longer content -> more visibility).
  VERDICT: MIXED/WEAK

Signal 3: active_days_h1 vs avg_position_h1 correlation = -0.0588
  Expect NEGATIVE if the assumption holds (more consistent visibility -> better/lower position number).
  VERDICT: MIXED/WEAK


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's own optimization flags include **"Fix CTR"** — its implicit assumption is that some pages are *already ranking well enough* (decent position) but *underperforming on clicks* (low CTR relative to that position), making them fixable through snippet/title work rather than a ranking problem. The test: do low-CTR pages in this portfolio actually tend to have workable positions, or are most of them *also* poorly positioned — in which case "Fix CTR" would be misdiagnosing a ranking problem as a copy problem?

In [3]:
# Define "low CTR" as bottom quartile of ctr_h1 among pages with real impressions, and check
# their position distribution against the rest of the population.
has_impressions = page_level_full[page_level_full["avg_impressions_h1"] > 0].copy()
ctr_p25 = has_impressions["ctr_h1"].quantile(0.25)

low_ctr = has_impressions[has_impressions["ctr_h1"] <= ctr_p25]
rest = has_impressions[has_impressions["ctr_h1"] > ctr_p25]

print(f"Low-CTR group (bottom 25%, n={len(low_ctr):,}): avg position = {low_ctr['avg_position_h1'].mean():.2f}")
print(f"Rest of population (n={len(rest):,}):            avg position = {rest['avg_position_h1'].mean():.2f}")

# "Fixable" definition: position under 20 (roughly page 1-2) despite low CTR - a real candidate
# for a snippet/title fix rather than a fundamentally weak ranking.
fixable_share = (low_ctr["avg_position_h1"] <= 20).mean()
print(f"\nShare of the low-CTR group that's ALSO reasonably well-positioned (avg_position_h1 <= 20): "
      f"{fixable_share:.1%}")
print("If this share is high, the 'Fix CTR' flag's assumption holds - these pages really are")
print("snippet/title problems, not ranking problems. If it's low, most 'low CTR' pages are")
print("actually also poorly ranked, and the flag may be misdiagnosing a chunk of its targets.")

Low-CTR group (bottom 25%, n=98,684): avg position = 18.67
Rest of population (n=51,849):            avg position = 10.32

Share of the low-CTR group that's ALSO reasonably well-positioned (avg_position_h1 <= 20): 69.8%
If this share is high, the 'Fix CTR' flag's assumption holds - these pages really are
snippet/title problems, not ranking problems. If it's low, most 'low CTR' pages are
actually also poorly ranked, and the flag may be misdiagnosing a chunk of its targets.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

To be finalized after the real run above — written honestly against the actual verdicts, not decided in advance. As a placeholder structure: if Signal 1 and the flag-linked test both confirm, a content team can trust "Fix CTR" as a real, distinct action bucket separate from "needs a full refresh." If the flag-linked test comes back MIXED or OPPOSITE, the practical takeaway is that the flag's fixable/non-fixable split needs a position cutoff added to it, since low CTR alone doesn't reliably separate a copy problem from a ranking problem in this data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.